# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_01 — Vectorized Backtest Engine

### Research question

How do we build a reusable, vectorized multi-asset backtesting engine that handles signal timing, position construction, execution lag, transaction costs, borrow/financing costs, NAV construction, performance analytics, leakage checks, and audit trails?

This notebook opens **Phase 5**. Earlier phases built signals, pricing models, optimisers, risk models, hedges, and allocation methods. Phase 5 is about whether those ideas survive contact with realistic validation.

This notebook covers:

1. synthetic multi-asset data generation;
2. price, return, spread, volume, and borrow-fee matrices;
3. research signal construction;
4. signal-to-weight conversion;
5. long-short and long-only strategies;
6. rebalance schedules;
7. execution lag;
8. transaction costs;
9. borrow and financing costs;
10. NAV accounting;
11. performance metrics;
12. drawdowns;
13. asset and class attribution;
14. exposure diagnostics;
15. leakage diagnostics;
16. proper lag versus no-lag comparison;
17. vectorized versus loop parity test;
18. stress-day diagnostics;
19. capacity proxy;
20. reusable engine class;
21. governance flags;
22. audit report;
23. saved outputs and manifest.

The key idea:

> A backtest is not a chart of cumulative returns. It is an accounting engine with timing rules, costs, constraints, validation checks, and auditability.

## 1. Timing convention

The most important rule:

$$
w^{held}_t = w^{target}_{t-\ell}
$$

where $\ell$ is the execution lag.

For a one-day close-to-close strategy, a signal observed at close $t$ can only be used for returns from $t+1$ onward.

Portfolio gross return:

$$
R^p_t = \sum_i w^{held}_{i,t}R_{i,t}
$$

Net return:

$$
\begin{aligned}
R^{net}_t &= R^p_t \\
&\quad - cost_t \\
&\quad - borrow_t \\
&\quad - financing_t
\end{aligned}
$$

Any backtest that violates this timing rule is not measuring a tradable strategy.

## 2. Cost model

Asset-level turnover:

$$
TO_{i,t}=|w^{held}_{i,t}-w^{held}_{i,t-1}|
$$

Transaction cost:

$$
cost_t = \sum_i TO_{i,t} \frac{\frac{spread_{i,t}}{2}+slippage_i}{10000}
$$

Short borrow cost:

$$
borrow_t = \sum_i \max(-w_{i,t},0)\frac{fee_i}{252}
$$

Financing cost on leverage above 1:

$$
financing_t = \max(gross_t-1,0)\frac{r_f}{252}
$$

The exact model is simplified, but the accounting structure is the important part.

## 3. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class BacktestEngineConfig:
    n_days: int = 1_800
    annualisation: int = 252
    seed: int = 42
    gross_target_long_short: float = 1.50
    gross_target_long_only: float = 1.00
    max_abs_weight: float = 0.20
    rebalance_step: int = 5
    execution_lag: int = 1
    base_slippage_bps: float = 2.0
    financing_rate_ann: float = 0.035
    risk_free_rate_ann: float = 0.025
    cvar_alpha: float = 0.95
    initial_capital: float = 1_000_000.0

config = BacktestEngineConfig()
config

## 4. Synthetic multi-asset market panel

The engine needs aligned matrices:

- returns;
- prices;
- spreads;
- dollar volumes;
- borrow fees;
- metadata;
- benchmark returns.

The simulation includes regimes and stress episodes so the diagnostics have something useful to inspect.

In [ ]:
def simulate_market_panel(config: BacktestEngineConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2017-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ", "US_BOND", "EU_BOND", "GOLD",
        "OIL", "COPPER", "FX_CARRY", "TREND", "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity", "EU_EQ": "equity", "EM_EQ": "equity",
        "US_BOND": "bond", "EU_BOND": "bond",
        "GOLD": "commodity", "OIL": "commodity", "COPPER": "commodity",
        "FX_CARRY": "fx", "TREND": "alternative",
        "REIT": "real_asset", "BTC_PROXY": "crypto",
    }

    base_spread_bps = pd.Series({
        "US_EQ": 1.0, "EU_EQ": 1.5, "EM_EQ": 4.0,
        "US_BOND": 0.8, "EU_BOND": 1.0,
        "GOLD": 2.0, "OIL": 3.5, "COPPER": 3.0,
        "FX_CARRY": 2.5, "TREND": 5.0, "REIT": 3.0, "BTC_PROXY": 12.0,
    })

    annual_borrow_fee = pd.Series({
        "US_EQ": 0.003, "EU_EQ": 0.004, "EM_EQ": 0.010,
        "US_BOND": 0.002, "EU_BOND": 0.002,
        "GOLD": 0.004, "OIL": 0.006, "COPPER": 0.006,
        "FX_CARRY": 0.007, "TREND": 0.012, "REIT": 0.008, "BTC_PROXY": 0.030,
    })

    adv_usd = pd.Series({
        "US_EQ": 8e9, "EU_EQ": 4e9, "EM_EQ": 2e9,
        "US_BOND": 10e9, "EU_BOND": 5e9,
        "GOLD": 3e9, "OIL": 2.5e9, "COPPER": 1.2e9,
        "FX_CARRY": 3e9, "TREND": 0.8e9, "REIT": 1.5e9, "BTC_PROXY": 1.0e9,
    })

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_ann = np.array([0.14, 0.055, 0.13, 0.08, 0.09, 0.40])
    factor_vol_daily = factor_vol_ann / np.sqrt(config.annualisation)

    factor_corr = np.array([
        [ 1.00, -0.25,  0.25,  0.15, -0.12,  0.35],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20],
        [ 0.25,  0.10,  1.00,  0.10,  0.05,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15],
        [-0.12,  0.15,  0.05,  0.00,  1.00,  0.00],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00],
    ])
    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.70]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "market"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.40
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.30

    annual_mean = pd.Series({
        "US_EQ": 0.070, "EU_EQ": 0.060, "EM_EQ": 0.080,
        "US_BOND": 0.025, "EU_BOND": 0.020,
        "GOLD": 0.035, "OIL": 0.045, "COPPER": 0.040,
        "FX_CARRY": 0.030, "TREND": 0.050, "REIT": 0.060, "BTC_PROXY": 0.120,
    })

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07, "EU_EQ": 0.08, "EM_EQ": 0.13,
        "US_BOND": 0.025, "EU_BOND": 0.030,
        "GOLD": 0.12, "OIL": 0.22, "COPPER": 0.18,
        "FX_CARRY": 0.07, "TREND": 0.08, "REIT": 0.11, "BTC_PROXY": 0.52,
    })

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    factor_returns = np.zeros((config.n_days, len(factor_names)))
    asset_returns = np.zeros((config.n_days, len(assets)))

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.4
        cov_t = factor_cov * vol_multiplier**2
        z = rng.standard_t(df=6, size=len(factor_names)) * np.sqrt((6 - 2) / 6)
        f = np.linalg.cholesky(cov_t + 1e-12 * np.eye(len(factor_names))) @ z

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
            f[4] += rng.uniform(0.005, 0.040)
        elif u < 0.014:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.035)
            f[2] += rng.uniform(0.030, 0.090)
            f[0] -= rng.uniform(0.010, 0.050)
        elif u < 0.020:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=5, size=len(assets)) * np.sqrt((5 - 2) / 5)
        eps = eps * (idio_vol_ann.values / np.sqrt(config.annualisation))

        predictive_tilt = np.zeros(len(assets))
        if t > 0 and regime[t - 1] == 1:
            predictive_tilt[assets.index("TREND")] += 0.0006
            predictive_tilt[assets.index("FX_CARRY")] -= 0.0004

        asset_returns[t] = annual_mean.values / config.annualisation + loadings.to_numpy() @ f + eps + predictive_tilt
        factor_returns[t] = f

    returns = pd.DataFrame(asset_returns, index=dates, columns=assets)
    prices = 100.0 * np.exp(np.log1p(returns).cumsum())

    spread_noise = rng.lognormal(mean=0.0, sigma=0.20, size=returns.shape)
    spread_bps = pd.DataFrame(spread_noise * base_spread_bps.values, index=dates, columns=assets)
    spread_bps = spread_bps.multiply(1.0 + 0.75 * pd.Series(regime, index=dates), axis=0)

    volume_noise = rng.lognormal(mean=0.0, sigma=0.45, size=returns.shape)
    dollar_volume = pd.DataFrame(volume_noise * adv_usd.values, index=dates, columns=assets)
    dollar_volume = dollar_volume.multiply(1.0 - 0.25 * pd.Series(regime, index=dates), axis=0)

    borrow_fee = pd.DataFrame(np.tile(annual_borrow_fee.values, (config.n_days, 1)), index=dates, columns=assets)

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "base_spread_bps": [base_spread_bps[a] for a in assets],
        "annual_borrow_fee": [annual_borrow_fee[a] for a in assets],
        "adv_usd": [adv_usd[a] for a in assets],
        "annual_mean_assumption": [annual_mean[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
    })

    factors = pd.DataFrame(factor_returns, index=dates, columns=factor_names)
    regime_info = pd.DataFrame({"regime": regime, "stress_type": stress_type}, index=dates)
    benchmark = 0.60 * returns["US_EQ"] + 0.25 * returns["US_BOND"] + 0.15 * returns["GOLD"]

    return returns, prices, spread_bps, dollar_volume, borrow_fee, metadata, factors, regime_info, benchmark

returns, prices, spread_bps, dollar_volume, borrow_fee, metadata, factors, regime_info, benchmark = simulate_market_panel(config)
asset_cols = metadata["asset"].tolist()

returns.head()

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(prices.index, prices[asset], label=asset, alpha=0.75)
plt.title("Synthetic Prices")
plt.xlabel("Date")
plt.ylabel("Price index")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(regime_info.index, regime_info["regime"], drawstyle="steps-post")
plt.title("Regime Indicator")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 5. Signal generation

We create a composite signal from:

1. 63-day momentum;
2. 21-day reversal;
3. static carry;
4. volatility penalty.

We also create an intentionally leaky signal using future returns, purely to test leakage diagnostics.

In [ ]:
def cross_sectional_zscore(df):
    mean = df.mean(axis=1)
    std = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mean, axis=0).div(std, axis=0).fillna(0.0)

def rolling_volatility(returns, window=63, annualisation=252):
    vol = returns.rolling(window).std()
    vol = vol.fillna(returns.expanding().std())
    return vol.fillna(0.0) * np.sqrt(annualisation)

def make_research_signals(returns, prices):
    momentum_63 = cross_sectional_zscore(prices.pct_change(63))
    reversal_21 = cross_sectional_zscore(-prices.pct_change(21))
    vol_penalty = cross_sectional_zscore(-rolling_volatility(returns, 63, config.annualisation))

    carry_static = pd.Series({
        "US_EQ": 0.10, "EU_EQ": 0.05, "EM_EQ": 0.12,
        "US_BOND": 0.02, "EU_BOND": 0.01,
        "GOLD": 0.00, "OIL": 0.06, "COPPER": 0.04,
        "FX_CARRY": 0.15, "TREND": 0.03, "REIT": 0.07, "BTC_PROXY": 0.20,
    }).reindex(returns.columns).fillna(0.0)

    carry = pd.DataFrame(np.tile(carry_static.values, (len(returns), 1)), index=returns.index, columns=returns.columns)
    carry = cross_sectional_zscore(carry)

    composite = 0.55 * momentum_63 + 0.20 * reversal_21 + 0.20 * carry + 0.25 * vol_penalty
    composite = composite.clip(lower=-3, upper=3).fillna(0.0)

    leaky = cross_sectional_zscore(returns.shift(-1)).fillna(0.0)

    return {
        "momentum_63": momentum_63,
        "reversal_21": reversal_21,
        "carry": carry,
        "vol_penalty": vol_penalty,
        "composite": composite,
        "leaky_future_return_signal": leaky,
    }

signals = make_research_signals(returns, prices)
signals["composite"].tail()

In [ ]:
plt.figure(figsize=(12, 5))
for asset in ["US_EQ", "US_BOND", "GOLD", "TREND", "BTC_PROXY"]:
    plt.plot(signals["composite"].index, signals["composite"][asset], label=asset, alpha=0.8)
plt.title("Composite Signal")
plt.xlabel("Date")
plt.ylabel("Cross-sectional signal")
plt.legend()
plt.show()

## 6. Signal-to-weight conversion

The research engine separates signal construction from portfolio construction.

Long-short construction:

- demean signals;
- scale by inverse volatility;
- normalise to gross exposure;
- cap position sizes.

Long-only construction:

- keep positive scores;
- scale by inverse volatility;
- normalise to 100% invested;
- cap position sizes.

In [ ]:
def cap_and_renormalise_weights(weights, gross_target, max_abs_weight, long_only=False, max_iter=10):
    w = weights.copy().fillna(0.0)

    for _ in range(max_iter):
        if long_only:
            w = w.clip(lower=0.0, upper=max_abs_weight)
            total = w.sum(axis=1).replace(0.0, np.nan)
            w = w.div(total, axis=0).fillna(0.0)
        else:
            w = w.clip(lower=-max_abs_weight, upper=max_abs_weight)
            gross = w.abs().sum(axis=1).replace(0.0, np.nan)
            w = w.div(gross, axis=0).mul(gross_target).fillna(0.0)

    return w

def signal_to_target_weights(signal, returns, gross_target, max_abs_weight, long_only=False, vol_window=63):
    vol = returns.rolling(vol_window).std().shift(1)
    vol = vol.fillna(returns.expanding().std().shift(1)).fillna(returns.std())
    inv_vol = 1.0 / vol.replace(0.0, np.nan)

    raw_signal = signal.copy().fillna(0.0)

    if long_only:
        score = raw_signal.sub(raw_signal.min(axis=1), axis=0)
        raw_weights = score.mul(inv_vol, axis=0).clip(lower=0.0)
        weights = raw_weights.div(raw_weights.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
        return cap_and_renormalise_weights(weights, gross_target, max_abs_weight, long_only=True)

    score = raw_signal.sub(raw_signal.mean(axis=1), axis=0)
    raw_weights = score.mul(inv_vol, axis=0)
    weights = raw_weights.div(raw_weights.abs().sum(axis=1).replace(0.0, np.nan), axis=0).mul(gross_target).fillna(0.0)
    return cap_and_renormalise_weights(weights, gross_target, max_abs_weight, long_only=False)

def apply_rebalance_schedule(target_weights, rebalance_step):
    if rebalance_step <= 1:
        return target_weights.copy().fillna(0.0)
    scheduled = target_weights.copy() * np.nan
    scheduled.iloc[::rebalance_step] = target_weights.iloc[::rebalance_step]
    return scheduled.ffill().fillna(0.0)

target_weights_ls = apply_rebalance_schedule(
    signal_to_target_weights(signals["composite"], returns, config.gross_target_long_short, config.max_abs_weight, long_only=False),
    config.rebalance_step
)

target_weights_lo = apply_rebalance_schedule(
    signal_to_target_weights(signals["composite"], returns, config.gross_target_long_only, config.max_abs_weight, long_only=True),
    config.rebalance_step
)

target_weights_leaky = apply_rebalance_schedule(
    signal_to_target_weights(signals["leaky_future_return_signal"], returns, config.gross_target_long_short, config.max_abs_weight, long_only=False),
    config.rebalance_step
)

target_weights_ls.tail()

## 7. Input validation

The engine checks alignment before running a strategy.

In [ ]:
def validate_backtest_inputs(returns, target_weights, spread_bps, borrow_fee, metadata):
    checks = [
        {
            "check": "returns_index_unique",
            "passed": bool(returns.index.is_unique),
            "detail": f"{returns.index[0]} to {returns.index[-1]}"
        },
        {
            "check": "weights_index_matches_returns",
            "passed": bool(target_weights.index.equals(returns.index)),
            "detail": f"{len(target_weights)} rows"
        },
        {
            "check": "weights_columns_match_returns",
            "passed": bool(list(target_weights.columns) == list(returns.columns)),
            "detail": f"{len(target_weights.columns)} assets"
        },
        {
            "check": "spread_index_columns_match",
            "passed": bool(spread_bps.index.equals(returns.index) and list(spread_bps.columns) == list(returns.columns)),
            "detail": "spread matrix"
        },
        {
            "check": "borrow_fee_index_columns_match",
            "passed": bool(borrow_fee.index.equals(returns.index) and list(borrow_fee.columns) == list(returns.columns)),
            "detail": "borrow fee matrix"
        },
        {
            "check": "returns_are_finite",
            "passed": bool(np.isfinite(returns.to_numpy()).all()),
            "detail": f"missing values: {int(returns.isna().sum().sum())}"
        },
        {
            "check": "metadata_covers_all_assets",
            "passed": bool(set(returns.columns).issubset(set(metadata["asset"]))),
            "detail": f"metadata assets: {metadata['asset'].nunique()}"
        },
    ]
    return pd.DataFrame(checks)

## 8. Vectorized backtest engine

This is the core engine.

It returns:

- strategy return table;
- held weights;
- target weights;
- delta weights;
- turnover matrix;
- validation report.

In [ ]:
def run_vectorized_backtest(
    returns,
    target_weights,
    spread_bps,
    borrow_fee,
    metadata,
    benchmark=None,
    execution_lag=1,
    base_slippage_bps=2.0,
    financing_rate_ann=0.035,
    annualisation=252,
):
    target_weights = target_weights.reindex(index=returns.index, columns=returns.columns).fillna(0.0)
    validation = validate_backtest_inputs(returns, target_weights, spread_bps, borrow_fee, metadata)

    held_weights = target_weights.shift(execution_lag).fillna(0.0)

    gross_return = (held_weights * returns).sum(axis=1)

    delta_weights = held_weights.diff().fillna(held_weights)
    turnover_asset = delta_weights.abs()
    turnover = turnover_asset.sum(axis=1)

    half_spread_bps = spread_bps.reindex_like(returns).ffill().fillna(0.0) / 2.0
    all_in_cost_bps = half_spread_bps + base_slippage_bps
    transaction_cost = (turnover_asset * all_in_cost_bps / 10000.0).sum(axis=1)

    short_weights = held_weights.clip(upper=0.0).abs()
    borrow_daily = borrow_fee.reindex_like(returns).ffill().fillna(0.0) / annualisation
    borrow_cost = (short_weights * borrow_daily).sum(axis=1)

    gross_exposure = held_weights.abs().sum(axis=1)
    net_exposure = held_weights.sum(axis=1)
    long_exposure = held_weights.clip(lower=0.0).sum(axis=1)
    short_exposure = held_weights.clip(upper=0.0).sum(axis=1)

    financing_cost = np.maximum(gross_exposure - 1.0, 0.0) * financing_rate_ann / annualisation

    net_return = gross_return - transaction_cost - borrow_cost - financing_cost
    nav = (1.0 + net_return).cumprod()

    result = pd.DataFrame({
        "gross_return": gross_return,
        "transaction_cost": transaction_cost,
        "borrow_cost": borrow_cost,
        "financing_cost": financing_cost,
        "net_return": net_return,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": gross_exposure,
        "net_exposure": net_exposure,
        "long_exposure": long_exposure,
        "short_exposure": short_exposure,
    }, index=returns.index)

    if benchmark is not None:
        result["benchmark_return"] = benchmark.reindex(returns.index).fillna(0.0)
        result["benchmark_nav"] = (1.0 + result["benchmark_return"]).cumprod()
        result["active_return"] = result["net_return"] - result["benchmark_return"]

    details = {
        "held_weights": held_weights,
        "target_weights": target_weights,
        "delta_weights": delta_weights,
        "turnover_asset": turnover_asset,
        "validation": validation,
    }

    return result, details

## 9. Run strategy backtests

In [ ]:
ls_result, ls_details = run_vectorized_backtest(
    returns, target_weights_ls, spread_bps, borrow_fee, metadata,
    benchmark=benchmark,
    execution_lag=config.execution_lag,
    base_slippage_bps=config.base_slippage_bps,
    financing_rate_ann=config.financing_rate_ann,
    annualisation=config.annualisation,
)

lo_result, lo_details = run_vectorized_backtest(
    returns, target_weights_lo, spread_bps, borrow_fee, metadata,
    benchmark=benchmark,
    execution_lag=config.execution_lag,
    base_slippage_bps=config.base_slippage_bps,
    financing_rate_ann=config.financing_rate_ann,
    annualisation=config.annualisation,
)

leaky_result, leaky_details = run_vectorized_backtest(
    returns, target_weights_leaky, spread_bps, borrow_fee, metadata,
    benchmark=benchmark,
    execution_lag=config.execution_lag,
    base_slippage_bps=config.base_slippage_bps,
    financing_rate_ann=config.financing_rate_ann,
    annualisation=config.annualisation,
)

equal_weight_target = pd.DataFrame(1.0 / len(asset_cols), index=returns.index, columns=returns.columns)

equal_weight_result, equal_weight_details = run_vectorized_backtest(
    returns, equal_weight_target, spread_bps, borrow_fee, metadata,
    benchmark=benchmark,
    execution_lag=config.execution_lag,
    base_slippage_bps=config.base_slippage_bps,
    financing_rate_ann=config.financing_rate_ann,
    annualisation=config.annualisation,
)

ls_details["validation"]

## 10. Performance metrics

In [ ]:
def max_drawdown(nav):
    running_max = nav.cummax()
    drawdown = nav / running_max - 1.0
    return drawdown, float(drawdown.min())

def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)

def alpha_beta(strategy_returns, benchmark_returns, annualisation):
    joined = pd.concat([strategy_returns.rename("strategy"), benchmark_returns.rename("benchmark")], axis=1).dropna()
    if len(joined) < 5 or joined["benchmark"].var(ddof=1) == 0:
        return np.nan, np.nan, np.nan
    beta = joined["strategy"].cov(joined["benchmark"]) / joined["benchmark"].var(ddof=1)
    alpha_daily = joined["strategy"].mean() - beta * joined["benchmark"].mean()
    corr = joined["strategy"].corr(joined["benchmark"])
    return float(alpha_daily * annualisation), float(beta), float(corr)

def performance_summary(result, name, annualisation=252, cvar_alpha=0.95):
    r = result["net_return"].dropna()
    nav = (1.0 + r).cumprod()
    dd, mdd = max_drawdown(nav)

    ann_return = (1.0 + r).prod() ** (annualisation / len(r)) - 1.0
    ann_vol = r.std(ddof=1) * np.sqrt(annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = downside.std(ddof=1) * np.sqrt(annualisation)
    sortino = ann_return / downside_vol if downside_vol > 0 else np.nan
    calmar = ann_return / abs(mdd) if mdd < 0 else np.nan

    var, cvar = historical_var_cvar(-r, cvar_alpha)

    if "benchmark_return" in result.columns:
        alpha_ann, beta, corr = alpha_beta(r, result["benchmark_return"], annualisation)
    else:
        alpha_ann, beta, corr = np.nan, np.nan, np.nan

    return {
        "strategy": name,
        "annualised_return": float(ann_return),
        "annualised_vol": float(ann_vol),
        "sharpe_proxy": float(sharpe),
        "sortino_proxy": float(sortino),
        "max_drawdown": float(mdd),
        "calmar_proxy": float(calmar),
        "hit_rate": float((r > 0).mean()),
        "skew": float(r.skew()),
        "excess_kurtosis": float(r.kurtosis()),
        f"VaR_{int(cvar_alpha*100)}": float(var),
        f"CVaR_{int(cvar_alpha*100)}": float(cvar),
        "total_return": float(nav.iloc[-1] - 1.0),
        "mean_turnover": float(result["turnover"].mean()),
        "annualised_transaction_cost": float(result["transaction_cost"].mean() * annualisation),
        "annualised_borrow_cost": float(result["borrow_cost"].mean() * annualisation),
        "annualised_financing_cost": float(result["financing_cost"].mean() * annualisation),
        "mean_gross_exposure": float(result["gross_exposure"].mean()),
        "mean_net_exposure": float(result["net_exposure"].mean()),
        "alpha_ann_vs_benchmark": float(alpha_ann),
        "beta_vs_benchmark": float(beta),
        "corr_vs_benchmark": float(corr),
    }

performance_table = pd.DataFrame([
    performance_summary(ls_result, "long_short_composite", config.annualisation, config.cvar_alpha),
    performance_summary(lo_result, "long_only_composite", config.annualisation, config.cvar_alpha),
    performance_summary(equal_weight_result, "equal_weight_baseline", config.annualisation, config.cvar_alpha),
    performance_summary(leaky_result, "leaky_future_return_signal", config.annualisation, config.cvar_alpha),
]).sort_values("sharpe_proxy", ascending=False)

performance_table

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(ls_result.index, ls_result["nav"], label="long-short composite")
plt.plot(lo_result.index, lo_result["nav"], label="long-only composite")
plt.plot(equal_weight_result.index, equal_weight_result["nav"], label="equal-weight baseline")
plt.plot(leaky_result.index, leaky_result["nav"], label="leaky signal", alpha=0.65)
plt.plot(ls_result.index, ls_result["benchmark_nav"], label="benchmark", linestyle="--")
plt.title("Strategy NAV Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 11. Drawdown report

In [ ]:
def drawdown_table(result, strategy_name, top_n=10):
    r = result["net_return"]
    nav = (1.0 + r).cumprod()
    dd, _ = max_drawdown(nav)

    rows = []
    in_dd = False
    start = None
    trough = None
    trough_dd = 0.0

    for date, value in dd.items():
        if value < 0 and not in_dd:
            in_dd = True
            start = date
            trough = date
            trough_dd = value
        elif value < 0 and in_dd:
            if value < trough_dd:
                trough = date
                trough_dd = value
        elif value == 0 and in_dd:
            rows.append({
                "strategy": strategy_name,
                "start": start,
                "trough": trough,
                "recovery": date,
                "max_drawdown": trough_dd,
                "days_to_trough": (trough - start).days,
                "days_to_recovery": (date - start).days,
            })
            in_dd = False

    if in_dd:
        rows.append({
            "strategy": strategy_name,
            "start": start,
            "trough": trough,
            "recovery": pd.NaT,
            "max_drawdown": trough_dd,
            "days_to_trough": (trough - start).days,
            "days_to_recovery": np.nan,
        })

    return pd.DataFrame(rows).sort_values("max_drawdown").head(top_n)

drawdown_report = pd.concat([
    drawdown_table(ls_result, "long_short_composite"),
    drawdown_table(lo_result, "long_only_composite"),
    drawdown_table(equal_weight_result, "equal_weight_baseline"),
], ignore_index=True)

drawdown_report

In [ ]:
plt.figure(figsize=(12, 6))
for result, label in [
    (ls_result, "long-short"),
    (lo_result, "long-only"),
    (equal_weight_result, "equal-weight"),
]:
    nav = (1 + result["net_return"]).cumprod()
    dd, _ = max_drawdown(nav)
    plt.plot(dd.index, dd, label=label)
plt.title("Drawdowns")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.show()

## 12. Attribution

Return contribution by asset:

$$
contribution_{i,t}=w^{held}_{i,t}R_{i,t}
$$

Then aggregate by asset and asset class.

In [ ]:
def contribution_attribution(returns, held_weights, metadata, strategy_name):
    contribution = held_weights * returns

    asset_rows = []
    for asset in returns.columns:
        asset_rows.append({
            "strategy": strategy_name,
            "asset": asset,
            "mean_daily_contribution": contribution[asset].mean(),
            "annualised_contribution": contribution[asset].mean() * config.annualisation,
            "vol_contribution_proxy": contribution[asset].std(ddof=1) * np.sqrt(config.annualisation),
            "average_weight": held_weights[asset].mean(),
            "average_abs_weight": held_weights[asset].abs().mean(),
        })

    asset_attr = pd.DataFrame(asset_rows).merge(metadata[["asset", "asset_class"]], on="asset", how="left")

    class_attr = (
        asset_attr
        .groupby(["strategy", "asset_class"])
        .agg(
            annualised_contribution=("annualised_contribution", "sum"),
            average_abs_weight=("average_abs_weight", "sum")
        )
        .reset_index()
        .sort_values("annualised_contribution", ascending=False)
    )

    return asset_attr, class_attr

ls_asset_attr, ls_class_attr = contribution_attribution(returns, ls_details["held_weights"], metadata, "long_short_composite")
lo_asset_attr, lo_class_attr = contribution_attribution(returns, lo_details["held_weights"], metadata, "long_only_composite")

asset_attribution = pd.concat([ls_asset_attr, lo_asset_attr], ignore_index=True)
class_attribution = pd.concat([ls_class_attr, lo_class_attr], ignore_index=True)

asset_attribution.sort_values("annualised_contribution", ascending=False).head(10)

In [ ]:
plot_df = ls_asset_attr.sort_values("annualised_contribution")

plt.figure(figsize=(10, 6))
plt.barh(plot_df["asset"], plot_df["annualised_contribution"])
plt.axvline(0, linestyle="--")
plt.title("Long-Short Composite: Annualised Contribution by Asset")
plt.xlabel("Annualised contribution")
plt.ylabel("Asset")
plt.show()

## 13. Exposure diagnostics

In [ ]:
exposure_summary = pd.DataFrame([
    {
        "strategy": "long_short_composite",
        "mean_gross": ls_result["gross_exposure"].mean(),
        "p95_gross": ls_result["gross_exposure"].quantile(0.95),
        "mean_net": ls_result["net_exposure"].mean(),
        "mean_long": ls_result["long_exposure"].mean(),
        "mean_short": ls_result["short_exposure"].mean(),
        "mean_turnover": ls_result["turnover"].mean(),
        "annual_cost_drag": (ls_result["transaction_cost"] + ls_result["borrow_cost"] + ls_result["financing_cost"]).mean() * config.annualisation,
    },
    {
        "strategy": "long_only_composite",
        "mean_gross": lo_result["gross_exposure"].mean(),
        "p95_gross": lo_result["gross_exposure"].quantile(0.95),
        "mean_net": lo_result["net_exposure"].mean(),
        "mean_long": lo_result["long_exposure"].mean(),
        "mean_short": lo_result["short_exposure"].mean(),
        "mean_turnover": lo_result["turnover"].mean(),
        "annual_cost_drag": (lo_result["transaction_cost"] + lo_result["borrow_cost"] + lo_result["financing_cost"]).mean() * config.annualisation,
    },
])

exposure_summary

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(ls_result.index, ls_result["gross_exposure"], label="gross exposure")
plt.plot(ls_result.index, ls_result["net_exposure"], label="net exposure")
plt.plot(ls_result.index, ls_result["turnover"].rolling(21).mean(), label="21d avg turnover")
plt.title("Long-Short Exposure Diagnostics")
plt.xlabel("Date")
plt.ylabel("Exposure / turnover")
plt.legend()
plt.show()

## 14. Leakage diagnostics

We compute cross-sectional information coefficients:

$$
IC_\ell = corr(signal_t, return_{t+\ell})
$$

A signal with suspiciously strong same-day or future-return correlation is flagged.

In [ ]:
def mean_cross_sectional_ic(signal, returns, lag):
    aligned_returns = returns.shift(-lag)
    values = []

    for date in signal.index:
        s = signal.loc[date]
        r = aligned_returns.loc[date]
        valid = s.notna() & r.notna()
        if valid.sum() >= 3:
            values.append(s[valid].corr(r[valid]))

    return pd.Series(values).dropna()

def leakage_diagnostic(signal_dict, returns):
    rows = []
    for name, sig in signal_dict.items():
        for lag in [-1, 0, 1, 2, 5]:
            ic = mean_cross_sectional_ic(sig, returns, lag=lag)
            rows.append({
                "signal": name,
                "lag": lag,
                "meaning": {
                    -1: "signal_t_vs_return_t_minus_1",
                    0: "signal_t_vs_return_t",
                    1: "signal_t_vs_return_t_plus_1",
                    2: "signal_t_vs_return_t_plus_2",
                    5: "signal_t_vs_return_t_plus_5",
                }[lag],
                "mean_ic": ic.mean(),
                "ic_ir": ic.mean() / ic.std(ddof=1) if ic.std(ddof=1) > 0 else np.nan,
                "abs_mean_ic": abs(ic.mean()),
                "n_obs": len(ic),
            })

    out = pd.DataFrame(rows)
    out["flag_suspicious"] = (
        ((out["lag"].isin([0, 1])) & (out["abs_mean_ic"] > 0.10))
        | ((out["signal"].str.contains("leaky")) & (out["lag"] == 1) & (out["abs_mean_ic"] > 0.05))
    )
    return out

leakage_report = leakage_diagnostic(
    {
        "composite": signals["composite"],
        "leaky_future_return_signal": signals["leaky_future_return_signal"],
    },
    returns
)

leakage_report

## 15. Proper lag versus no-lag comparison

A no-lag backtest is intentionally incorrect for this close-to-close setup.

If it looks much better, timing assumptions are dangerous.

In [ ]:
ls_no_lag_result, _ = run_vectorized_backtest(
    returns, target_weights_ls, spread_bps, borrow_fee, metadata,
    benchmark=benchmark,
    execution_lag=0,
    base_slippage_bps=config.base_slippage_bps,
    financing_rate_ann=config.financing_rate_ann,
    annualisation=config.annualisation,
)

timing_comparison = pd.DataFrame([
    performance_summary(ls_result, "proper_lag_1", config.annualisation, config.cvar_alpha),
    performance_summary(ls_no_lag_result, "incorrect_no_lag", config.annualisation, config.cvar_alpha),
])

timing_comparison

## 16. Vectorized versus loop parity test

A loop implementation is slower but easier to inspect.

The vectorized engine should match it to numerical precision.

In [ ]:
def run_loop_backtest(returns, target_weights, spread_bps, borrow_fee, execution_lag, base_slippage_bps, financing_rate_ann, annualisation):
    target = target_weights.reindex_like(returns).fillna(0.0)
    held = target.shift(execution_lag).fillna(0.0)

    rows = []
    prev_w = pd.Series(0.0, index=returns.columns)
    nav = 1.0

    for date in returns.index:
        w = held.loc[date]
        r = returns.loc[date]

        gross_return = float((w * r).sum())
        delta = (w - prev_w).abs()
        cost_bps = spread_bps.loc[date] / 2.0 + base_slippage_bps
        transaction_cost = float((delta * cost_bps / 10000.0).sum())

        short = w.clip(upper=0.0).abs()
        borrow_cost = float((short * borrow_fee.loc[date] / annualisation).sum())

        gross_exposure = float(w.abs().sum())
        financing_cost = max(gross_exposure - 1.0, 0.0) * financing_rate_ann / annualisation

        net_return = gross_return - transaction_cost - borrow_cost - financing_cost
        nav *= (1.0 + net_return)

        rows.append({
            "net_return": net_return,
            "nav": nav,
            "transaction_cost": transaction_cost,
            "borrow_cost": borrow_cost,
            "financing_cost": financing_cost,
            "turnover": float(delta.sum()),
            "gross_exposure": gross_exposure,
        })

        prev_w = w

    return pd.DataFrame(rows, index=returns.index)

loop_ls_result = run_loop_backtest(
    returns, target_weights_ls, spread_bps, borrow_fee,
    config.execution_lag,
    config.base_slippage_bps,
    config.financing_rate_ann,
    config.annualisation,
)

parity_report = pd.Series({
    "max_abs_net_return_difference": float((loop_ls_result["net_return"] - ls_result["net_return"]).abs().max()),
    "max_abs_nav_difference": float((loop_ls_result["nav"] - ls_result["nav"]).abs().max()),
    "mean_abs_net_return_difference": float((loop_ls_result["net_return"] - ls_result["net_return"]).abs().mean()),
})

parity_report

## 17. Stress-day diagnostics

In [ ]:
def stress_diagnostics(result, regime_info, strategy_name):
    joined = result.join(regime_info, how="left")

    by_stress = (
        joined
        .groupby("stress_type")
        .agg(
            n_days=("net_return", "count"),
            mean_return=("net_return", "mean"),
            vol=("net_return", "std"),
            worst_day=("net_return", "min"),
            avg_turnover=("turnover", "mean"),
            avg_cost=("transaction_cost", "mean"),
        )
        .reset_index()
    )
    by_stress["strategy"] = strategy_name
    by_stress["ann_mean_return"] = by_stress["mean_return"] * config.annualisation
    by_stress["ann_vol"] = by_stress["vol"] * np.sqrt(config.annualisation)

    worst_days = joined.nsmallest(10, "net_return")[
        ["net_return", "gross_return", "transaction_cost", "borrow_cost", "financing_cost",
         "turnover", "gross_exposure", "net_exposure", "regime", "stress_type"]
    ].copy()
    worst_days["strategy"] = strategy_name

    return by_stress, worst_days

ls_stress_summary, ls_worst_days = stress_diagnostics(ls_result, regime_info, "long_short_composite")
lo_stress_summary, lo_worst_days = stress_diagnostics(lo_result, regime_info, "long_only_composite")

stress_summary = pd.concat([ls_stress_summary, lo_stress_summary], ignore_index=True)
worst_days_report = pd.concat([ls_worst_days, lo_worst_days], axis=0)

stress_summary

## 18. Capacity proxy

A simple capacity check compares traded dollars to average daily dollar volume:

$$
participation_{i,t} = \frac{C|\Delta w_{i,t}|}{ADV_{i,t}}
$$

This is not a market-impact model, but it flags obviously impossible turnover.

In [ ]:
def capacity_report(delta_weights, dollar_volume, capital):
    trade_dollars = delta_weights.abs() * capital
    participation = trade_dollars / dollar_volume.replace(0.0, np.nan)

    rows = []
    for asset in delta_weights.columns:
        p = participation[asset].dropna()
        rows.append({
            "asset": asset,
            "mean_participation": p.mean(),
            "p95_participation": p.quantile(0.95),
            "max_participation": p.max(),
            "flag_p95_above_5pct_adv": bool(p.quantile(0.95) > 0.05),
            "flag_max_above_20pct_adv": bool(p.max() > 0.20),
        })

    return pd.DataFrame(rows).merge(metadata[["asset", "asset_class", "adv_usd"]], on="asset", how="left")

capacity_ls = capacity_report(ls_details["delta_weights"], dollar_volume, config.initial_capital)
capacity_lo = capacity_report(lo_details["delta_weights"], dollar_volume, config.initial_capital)

capacity_ls.sort_values("p95_participation", ascending=False)

## 19. Reusable engine class

This small wrapper stores results, details, and validation reports.

In [ ]:
class VectorizedBacktestEngine:
    def __init__(self, returns, spread_bps, borrow_fee, metadata, benchmark=None, config=None):
        self.returns = returns
        self.spread_bps = spread_bps
        self.borrow_fee = borrow_fee
        self.metadata = metadata
        self.benchmark = benchmark
        self.config = config
        self.results = {}
        self.details = {}

    def run(self, name, target_weights):
        cfg = self.config
        result, detail = run_vectorized_backtest(
            returns=self.returns,
            target_weights=target_weights,
            spread_bps=self.spread_bps,
            borrow_fee=self.borrow_fee,
            metadata=self.metadata,
            benchmark=self.benchmark,
            execution_lag=cfg.execution_lag,
            base_slippage_bps=cfg.base_slippage_bps,
            financing_rate_ann=cfg.financing_rate_ann,
            annualisation=cfg.annualisation,
        )
        self.results[name] = result
        self.details[name] = detail
        return result

    def summary(self):
        rows = []
        for name, result in self.results.items():
            rows.append(performance_summary(result, name, self.config.annualisation, self.config.cvar_alpha))
        return pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)

    def validation_report(self):
        frames = []
        for name, detail in self.details.items():
            v = detail["validation"].copy()
            v["strategy"] = name
            frames.append(v)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

engine = VectorizedBacktestEngine(
    returns=returns,
    spread_bps=spread_bps,
    borrow_fee=borrow_fee,
    metadata=metadata,
    benchmark=benchmark,
    config=config,
)

engine.run("long_short_composite", target_weights_ls)
engine.run("long_only_composite", target_weights_lo)
engine.run("equal_weight_baseline", equal_weight_target)

engine.summary()

## 20. Governance flags

A strategy should be reviewed if:

1. validation fails;
2. suspicious leakage appears;
3. no-lag performance is much better;
4. turnover is excessive;
5. cost drag is too high;
6. drawdown is too large;
7. capacity proxy is too high;
8. vectorized-loop parity fails.

In [ ]:
proper_sharpe = timing_comparison[timing_comparison["strategy"] == "proper_lag_1"]["sharpe_proxy"].iloc[0]
nolag_sharpe = timing_comparison[timing_comparison["strategy"] == "incorrect_no_lag"]["sharpe_proxy"].iloc[0]
ls_mdd = performance_table[performance_table["strategy"] == "long_short_composite"]["max_drawdown"].iloc[0]

governance_flags = pd.DataFrame([{
    "strategy": "long_short_composite",
    "validation_passed": bool(ls_details["validation"]["passed"].all()),
    "leakage_flags_count": int(leakage_report["flag_suspicious"].sum()),
    "proper_lag_sharpe": proper_sharpe,
    "no_lag_sharpe": nolag_sharpe,
    "no_lag_minus_proper_sharpe": nolag_sharpe - proper_sharpe,
    "mean_turnover": ls_result["turnover"].mean(),
    "annualised_total_cost_drag": (ls_result["transaction_cost"] + ls_result["borrow_cost"] + ls_result["financing_cost"]).mean() * config.annualisation,
    "max_drawdown": ls_mdd,
    "p95_max_participation": capacity_ls["p95_participation"].max(),
    "vectorized_loop_max_abs_return_diff": parity_report["max_abs_net_return_difference"],
    "flag_validation_failed": bool(not ls_details["validation"]["passed"].all()),
    "flag_suspicious_leakage": bool(leakage_report["flag_suspicious"].sum() > 0),
    "flag_no_lag_too_good": bool(nolag_sharpe - proper_sharpe > 0.75),
    "flag_turnover_above_50pct": bool(ls_result["turnover"].mean() > 0.50),
    "flag_cost_drag_above_5pct_ann": bool((ls_result["transaction_cost"] + ls_result["borrow_cost"] + ls_result["financing_cost"]).mean() * config.annualisation > 0.05),
    "flag_drawdown_below_minus_20pct": bool(ls_mdd < -0.20),
    "flag_capacity_p95_above_5pct_adv": bool(capacity_ls["p95_participation"].max() > 0.05),
    "flag_vectorized_loop_mismatch": bool(parity_report["max_abs_net_return_difference"] > 1e-12),
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_validation_failed",
        "flag_suspicious_leakage",
        "flag_no_lag_too_good",
        "flag_turnover_above_50pct",
        "flag_cost_drag_above_5pct_ann",
        "flag_drawdown_below_minus_20pct",
        "flag_capacity_p95_above_5pct_adv",
        "flag_vectorized_loop_mismatch",
    ]
].any(axis=1)

governance_flags

## 21. Audit report

In [ ]:
audit_rows = []

held_expected = target_weights_ls.shift(config.execution_lag).fillna(0.0)
held_diff = (held_expected - ls_details["held_weights"]).abs().max().max()
audit_rows.append({
    "check": "held_weights_equal_lagged_target",
    "value": float(held_diff),
    "passed": bool(held_diff < 1e-12),
})

nav_recalc = (1.0 + ls_result["net_return"]).cumprod()
nav_diff = (nav_recalc - ls_result["nav"]).abs().max()
audit_rows.append({
    "check": "nav_matches_cumprod_net_returns",
    "value": float(nav_diff),
    "passed": bool(nav_diff < 1e-12),
})

costs_nonnegative = bool((ls_result[["transaction_cost", "borrow_cost", "financing_cost"]] >= -1e-15).all().all())
audit_rows.append({
    "check": "costs_nonnegative",
    "value": costs_nonnegative,
    "passed": costs_nonnegative,
})

audit_rows.append({
    "check": "vectorized_loop_net_return_match",
    "value": float(parity_report["max_abs_net_return_difference"]),
    "passed": bool(parity_report["max_abs_net_return_difference"] < 1e-12),
})

audit_rows.append({
    "check": "input_validation_passed",
    "value": bool(ls_details["validation"]["passed"].all()),
    "passed": bool(ls_details["validation"]["passed"].all()),
})

finite_outputs = bool(np.isfinite(ls_result.to_numpy()).all())
audit_rows.append({
    "check": "result_outputs_finite",
    "value": finite_outputs,
    "passed": finite_outputs,
})

audit_rows.append({
    "check": "turnover_nonnegative",
    "value": bool((ls_result["turnover"] >= -1e-15).all()),
    "passed": bool((ls_result["turnover"] >= -1e-15).all()),
})

audit_rows.append({
    "check": "gross_exposure_nonnegative",
    "value": bool((ls_result["gross_exposure"] >= -1e-15).all()),
    "passed": bool((ls_result["gross_exposure"] >= -1e-15).all()),
})

audit_report = pd.DataFrame(audit_rows)
audit_report

## 22. Practical checklist for a backtest engine

Before trusting a backtest, check:

1. signal timestamp and availability;
2. execution lag;
3. rebalance rules;
4. costs and borrow assumptions;
5. financing and leverage;
6. gross/net exposure;
7. turnover;
8. benchmark comparison;
9. drawdowns and tail risk;
10. leakage diagnostics;
11. vectorized-loop parity;
12. capacity proxy;
13. reproducible config and manifest.

The best research teams do not just ask whether a strategy made money. They ask whether the result could have been known, traded, paid for, audited, and repeated.

## 23. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/vectorized_backtest_engine_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "returns": output_dir / "returns.csv",
    "prices": output_dir / "prices.csv",
    "spread_bps": output_dir / "spread_bps.csv",
    "dollar_volume": output_dir / "dollar_volume.csv",
    "borrow_fee": output_dir / "borrow_fee.csv",
    "metadata": output_dir / "metadata.csv",
    "factors": output_dir / "factors.csv",
    "regime_info": output_dir / "regime_info.csv",
    "benchmark": output_dir / "benchmark.csv",
    "signals_composite": output_dir / "signals_composite.csv",
    "signals_leaky": output_dir / "signals_leaky_future_return.csv",
    "target_weights_long_short": output_dir / "target_weights_long_short.csv",
    "target_weights_long_only": output_dir / "target_weights_long_only.csv",
    "held_weights_long_short": output_dir / "held_weights_long_short.csv",
    "backtest_long_short": output_dir / "backtest_long_short_composite.csv",
    "backtest_long_only": output_dir / "backtest_long_only_composite.csv",
    "backtest_equal_weight": output_dir / "backtest_equal_weight.csv",
    "backtest_leaky": output_dir / "backtest_leaky_signal.csv",
    "performance_table": output_dir / "performance_table.csv",
    "drawdown_report": output_dir / "drawdown_report.csv",
    "asset_attribution": output_dir / "asset_attribution.csv",
    "class_attribution": output_dir / "class_attribution.csv",
    "exposure_summary": output_dir / "exposure_summary.csv",
    "leakage_report": output_dir / "leakage_report.csv",
    "timing_comparison": output_dir / "timing_comparison.csv",
    "parity_report": output_dir / "vectorized_loop_parity_report.csv",
    "stress_summary": output_dir / "stress_summary.csv",
    "worst_days_report": output_dir / "worst_days_report.csv",
    "capacity_report": output_dir / "capacity_report_long_short.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

returns.to_csv(paths["returns"])
prices.to_csv(paths["prices"])
spread_bps.to_csv(paths["spread_bps"])
dollar_volume.to_csv(paths["dollar_volume"])
borrow_fee.to_csv(paths["borrow_fee"])
metadata.to_csv(paths["metadata"], index=False)
factors.to_csv(paths["factors"])
regime_info.to_csv(paths["regime_info"])
benchmark.to_frame("benchmark_return").to_csv(paths["benchmark"])

signals["composite"].to_csv(paths["signals_composite"])
signals["leaky_future_return_signal"].to_csv(paths["signals_leaky"])
target_weights_ls.to_csv(paths["target_weights_long_short"])
target_weights_lo.to_csv(paths["target_weights_long_only"])
ls_details["held_weights"].to_csv(paths["held_weights_long_short"])

ls_result.to_csv(paths["backtest_long_short"])
lo_result.to_csv(paths["backtest_long_only"])
equal_weight_result.to_csv(paths["backtest_equal_weight"])
leaky_result.to_csv(paths["backtest_leaky"])

performance_table.to_csv(paths["performance_table"], index=False)
drawdown_report.to_csv(paths["drawdown_report"], index=False)
asset_attribution.to_csv(paths["asset_attribution"], index=False)
class_attribution.to_csv(paths["class_attribution"], index=False)
exposure_summary.to_csv(paths["exposure_summary"], index=False)
leakage_report.to_csv(paths["leakage_report"], index=False)
timing_comparison.to_csv(paths["timing_comparison"], index=False)
parity_report.to_frame("value").to_csv(paths["parity_report"])
stress_summary.to_csv(paths["stress_summary"], index=False)
worst_days_report.to_csv(paths["worst_days_report"])
capacity_ls.to_csv(paths["capacity_report"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "vectorized_backtest_engine_outputs",
    "schema_version": "vectorized_backtest_engine_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "strategy_names": [
        "long_short_composite",
        "long_only_composite",
        "equal_weight_baseline",
        "leaky_future_return_signal",
    ],
    "performance_table": performance_table.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "validation_passed": bool(ls_details["validation"]["passed"].all()),
    "parity_report": parity_report.to_dict(),
    "important_timing_rule": "held_weights[t] = target_weights[t - execution_lag]",
    "notes": [
        "Signals are generated at close t and traded with execution_lag days.",
        "Transaction costs include half-spread and base slippage in basis points.",
        "Borrow cost is charged on short weights.",
        "Financing cost is charged on gross exposure above 1.",
        "Vectorized and loop backtests are compared for accounting parity.",
        "A deliberately leaky signal is included to demonstrate leakage diagnostics.",
        "Capacity proxy compares trade dollars to simulated ADV.",
        "This notebook is an engine template for later Phase 5 validation notebooks."
    ]
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["returns"], paths["backtest_long_short"], paths["performance_table"], paths["governance_flags"], paths["manifest"]

## 24. Limitations

### Synthetic data

The dataset is synthetic. Real backtests require adjusted prices, delistings, corporate actions, futures rolls, FX conversion, survivorship controls, and timestamped availability.

### Simple cost model

Costs are spread plus fixed slippage. Real transaction costs require impact, liquidity, participation, volatility, latency, and venue assumptions.

### Weight-based accounting

The engine uses portfolio weights. Production systems may require shares, contracts, margin, cash, collateral, dividends, tax lots, and futures multipliers.

### No intraday execution

The engine assumes daily close-to-close returns and delayed execution. Intraday strategies require event-driven simulation.

### Simple leakage checks

IC-based leakage diagnostics are smoke alarms, not formal proofs.

### No parameter search yet

This notebook avoids optimisation sweeps. Later notebooks should add walk-forward validation and overfitting diagnostics.

## 25. Research frontier and extensions

Important extensions include:

1. event-driven backtesting;
2. futures contract accounting;
3. corporate action handling;
4. realistic transaction-cost models;
5. purged cross-validation;
6. walk-forward validation;
7. parameter-search governance;
8. live shadow-mode monitoring;
9. dataset and config hashing;
10. Chinese futures support with contract rolls, night sessions, price limits, margin, tick size, exchange calendars, and L1 quote-based slippage.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_02_performance_attribution_report_template**  
   Turn raw results into a professional research report.

2. **05_03_transaction_costs_slippage_latency**  
   Build a more realistic cost engine.

3. **05_04_walk_forward_hedge_validation**  
   Validate hedge ratios and overlays.

4. **05_05_purged_cross_validation_backtesting**  
   Avoid leakage in model evaluation.

5. **05_06_walk_forward_model_validation**  
   Validate ML and signal models properly.

6. **05_10_research_audit_trail_and_manifest**  
   Make research reproducible and reviewable.

## 27. Summary

This notebook implemented a vectorized backtest engine.

It showed how to:

1. simulate a multi-asset research dataset;
2. generate realistic signals;
3. convert signals into target weights;
4. apply rebalance schedules;
5. enforce execution lag;
6. compute held weights;
7. calculate gross returns;
8. include transaction, borrow, and financing costs;
9. build NAV;
10. calculate performance metrics;
11. analyse drawdowns;
12. attribute returns by asset and class;
13. diagnose exposures and turnover;
14. detect suspicious leakage patterns;
15. compare proper lag with incorrect no-lag;
16. verify vectorized accounting against a loop implementation;
17. inspect stress-day behaviour;
18. estimate capacity through ADV participation;
19. wrap the logic into a reusable engine class;
20. save outputs, governance flags, and audit reports.

The key computational takeaway:

> Vectorization is not enough; the engine must preserve the correct sequence of information, decisions, trades, holdings, returns, costs, and reports.

The key financial takeaway:

> A strategy that only works before costs, without execution lag, or under leaky timing is not a strategy — it is a spreadsheet hallucination.

## 28. Further reading

- López de Prado, *Advances in Financial Machine Learning.*
- Bailey et al. on backtest overfitting.
- Grinold and Kahn, *Active Portfolio Management.*
- Chan, *Algorithmic Trading.*
- Cartea, Jaimungal, and Penalva, *Algorithmic and High-Frequency Trading.*
- Institutional research process literature on signal validation, transaction costs, capacity, and production monitoring.